##Посилання на csv файл з результатами розрахунків першого етапу завдання:
 https://drive.google.com/file/d/12E25Ztdrh2EA94pyfa8JefCMz_Uex2Ba/view?usp=sharing


##Посилання на дашборд в Tableau: https://public.tableau.com/views/ABTestSignificance_17790316790640/ConversionMetricsbyTest?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link

In [1]:
from google.colab import drive
from statsmodels.stats.proportion import proportions_ztest
import pandas as pd
import numpy as np

drive.mount('/content/drive')
%cd /content/drive/MyDrive/portfolio/Portfolio_Project_2

df = pd.read_csv("prepared_data.csv")

df['event_name'] = df['event_name'].astype(str).str.strip().str.lower()
df['event_name'] = df['event_name'].replace({
    'new accounts': 'new account'
})

metrics = {
    'add_payment_info / session': ('add_payment_info', 'session'),
    'add_shipping_info / session': ('add_shipping_info', 'session'),
    'begin_checkout / session': ('begin_checkout', 'session'),
    'new_accounts / session': ('new account', 'session')
}

results = []

for test_number in sorted(df['test'].dropna().unique()):
    test_data = df[df['test'] == test_number]

    for metric_name, (numerator_event, denominator_event) in metrics.items():
        control_data = test_data[test_data['test_group'] == 1]
        test_group_data = test_data[test_data['test_group'] == 2]

        denominator_count_control = control_data['ga_session_id'].nunique()
        denominator_count_test = test_group_data['ga_session_id'].nunique()

        numerator_count_control = control_data.loc[
            control_data['event_name'] == numerator_event, 'ga_session_id'
        ].nunique()

        numerator_count_test = test_group_data.loc[
            test_group_data['event_name'] == numerator_event, 'ga_session_id'
        ].nunique()

        conversion_rate_control = (
            numerator_count_control / denominator_count_control
            if denominator_count_control > 0 else np.nan
        )
        conversion_rate_test = (
            numerator_count_test / denominator_count_test
            if denominator_count_test > 0 else np.nan
        )

        if pd.notna(conversion_rate_control) and conversion_rate_control != 0:
            metric_change = ((conversion_rate_test - conversion_rate_control) / conversion_rate_control) * 100
        else:
            metric_change = np.nan

        if denominator_count_control > 0 and denominator_count_test > 0:
            z_stat, p_value = proportions_ztest(
                count=[numerator_count_test, numerator_count_control],
                nobs=[denominator_count_test, denominator_count_control],
                alternative='two-sided'
            )
        else:
            z_stat, p_value = np.nan, np.nan

        results.append({
            "test_number": int(test_number),
            "metric": metric_name,
            "numerator_event": numerator_event,
            "denominator_event": denominator_event,
            "numerator_count_test": int(numerator_count_test),
            "denominator_count_test": int(denominator_count_test),
            "conversion_rate_test": round(conversion_rate_test, 6) if pd.notna(conversion_rate_test) else np.nan,
            "numerator_count_control": int(numerator_count_control),
            "denominator_count_control": int(denominator_count_control),
            "conversion_rate_control": round(conversion_rate_control, 6) if pd.notna(conversion_rate_control) else np.nan,
            "metric_change_%": round(metric_change, 2) if pd.notna(metric_change) else np.nan,
            "z_stat": round(z_stat, 4) if pd.notna(z_stat) else np.nan,
            "p_value": round(p_value, 6) if pd.notna(p_value) else np.nan,
            "significant": bool(p_value < 0.05) if pd.notna(p_value) else False
        })

results_df = pd.DataFrame(results)
results_df.to_csv("ab_testing_results_corrected.csv", index=False)

print(results_df.head(10))
print(results_df['significant'].value_counts(dropna=False))


Mounted at /content/drive
/content/drive/MyDrive/portfolio/Portfolio_Project_2
   test_number                       metric    numerator_event  \
0            1   add_payment_info / session   add_payment_info   
1            1  add_shipping_info / session  add_shipping_info   
2            1     begin_checkout / session     begin_checkout   
3            1       new_accounts / session        new account   
4            2   add_payment_info / session   add_payment_info   
5            2  add_shipping_info / session  add_shipping_info   
6            2     begin_checkout / session     begin_checkout   
7            2       new_accounts / session        new account   
8            3   add_payment_info / session   add_payment_info   
9            3  add_shipping_info / session  add_shipping_info   

  denominator_event  numerator_count_test  denominator_count_test  \
0           session                    31                    5413   
1           session                    58               

У проєкті було проаналізовано 4 A/B‑тести та 4 ключові метрики конверсії на рівні сесій. Для кожної метрики розраховано конверсію в control і test‑групах, відносну зміну, z‑статистику та p‑value. За результатами аналізу жодна з метрик не показала статистично значущих змін на рівні значущості 0.05, тобто спостережені відмінності можуть бути наслідком випадкових коливань. Найбільші відносні зміни не супроводжувались низькими p‑value, тому їх не можна інтерпретувати як надійний ефект експерименту. Це може свідчити про слабкий вплив змін у test‑групах, недостатній обсяг вибірки або мінімальний вплив експерименту на поведінку користувачів.